# Task 4: Fine-Tuning BERT on Kaggle Dataset
This project demonstrates BERT fine-tuning for NLP classification tasks.

In [ ]:
!pip install transformers datasets torch accelerate -q

In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import torch

In [ ]:
data = {
    "text": [
        "I love this movie",
        "This product is amazing",
        "Worst experience ever",
        "I hate this service",
        "Absolutely fantastic",
        "Very disappointing",
        "Excellent quality",
        "Not worth the money",
        "Highly recommended",
        "Terrible customer support"
    ],

    "label": [1,1,0,0,1,0,1,0,1,0]
}

df = pd.DataFrame(data)

df.head()

In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    random_state=42
)

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
def tokenize_function(texts):
    return tokenizer(
        texts.tolist(),
        padding="max_length",
        truncation=True,
        max_length=64
    )

In [ ]:
train_encodings = tokenize_function(train_texts)
test_encodings = tokenize_function(test_texts)

In [ ]:
train_dataset = Dataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'labels': train_labels.tolist()
})

test_dataset = Dataset.from_dict({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask'],
    'labels': test_labels.tolist()
})

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    logging_dir='./logs',
    logging_steps=1
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [ ]:
trainer.train()

In [ ]:
predictions = trainer.predict(test_dataset)

preds = torch.argmax(
    torch.tensor(predictions.predictions),
    axis=1
)

accuracy = accuracy_score(test_labels, preds)

print("Model Accuracy:", accuracy)